In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Processor
import torch.nn as nn
import torch.optim as optim
import librosa
from torch.nn.utils.rnn import pad_sequence
import os
import re
import pandas as pd
from tqdm import tqdm
from processing import process_labels
from torch.cuda.amp import GradScaler, autocast

# 경로 설정
preprocessed_train_audio_path = "/home/wangan/Data/Training/preprocessed_audio_train/"
preprocessed_test_audio_path = "/home/wangan/Data/Validation/preprocessed_audio_test/"
train_label_path = "/home/wangan/final_data/label_traning.csv"
test_label_path = "/home/wangan/final_data/label_test.csv"

# 파일 경로 리스트 생성
pr_train = os.listdir(preprocessed_train_audio_path)
pr_test = os.listdir(preprocessed_test_audio_path)
pr_train_path = [os.path.join(preprocessed_train_audio_path, p) for p in pr_train]
pr_test_path = [os.path.join(preprocessed_test_audio_path, p) for p in pr_test]

# 정렬 함수 정의
def extract_sort_keys(file_path):
    id_match = re.search(r'/([\d]{4})-', file_path)
    file_num_match = re.search(r'-(\d+)\.mp3', file_path)
    id_key = id_match.group(1) if id_match else '0000'
    file_num_key = int(file_num_match.group(1)) if file_num_match else 0
    return id_key, file_num_key

# 훈련 데이터 정렬
df_train = pd.DataFrame({'file_path': pr_train_path, 'id': [re.search(r'/([\d]{4})-', path).group(1) for path in pr_train_path]})
df_train['sort_keys'] = df_train['file_path'].apply(extract_sort_keys)
df_sorted_train = df_train.sort_values(by='sort_keys').drop('sort_keys', axis=1).reset_index(drop=True)

# 테스트 데이터 정렬
df_test = pd.DataFrame({'file_path': pr_test_path, 'id': [re.search(r'/([\d]{4})-', path).group(1) for path in pr_test_path]})
df_test['sort_keys'] = df_test['file_path'].apply(extract_sort_keys)
df_sorted_test = df_test.sort_values(by='sort_keys').drop('sort_keys', axis=1).reset_index(drop=True)

# 데이터셋 클래스 정의
class AudioDataset(Dataset):
    def __init__(self, audio_files, labels, processor, sampling_rate=16000):
        self.audio_files = audio_files
        self.labels = labels
        self.processor = processor
        self.sampling_rate = sampling_rate

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, idx):
        audio_path = self.audio_files[idx]
        label = self.labels[idx]
        
        audio, _ = librosa.load(audio_path, sr=self.sampling_rate)
        input_values = self.processor(audio, sampling_rate=self.sampling_rate, return_tensors="pt").input_values
        
        return input_values.squeeze(0), torch.tensor(label, dtype=torch.long)

# Collate 함수 정의
def collate_fn(batch):
    inputs = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])
    inputs_padded = pad_sequence(inputs, batch_first=True)
    return inputs_padded, labels

# 모델 및 프로세서 설정 (use_cache 제거)
processor = Wav2Vec2Processor.from_pretrained("kresnik/wav2vec2-large-xlsr-korean")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "kresnik/wav2vec2-large-xlsr-korean", 
    num_labels=5,
    gradient_checkpointing=True  # 이 줄은 그대로 유지합니다.
)

for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

# 데이터셋 로드
train_audio_files = df_sorted_train['file_path'].to_list()
test_audio_files = df_sorted_test['file_path'].to_list()
train_labels = process_labels(train_label_path).to_list()
test_labels = process_labels(test_label_path).to_list()

train_dataset = AudioDataset(train_audio_files, train_labels, processor)
val_dataset = AudioDataset(test_audio_files, test_labels, processor)

# DataLoader 설정 변경 (배치 크기 증가)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

# 학습 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)
scaler = GradScaler()

# 학습 루프
accumulation_steps = 32  # 필요에 따라 조정 가능
num_epochs = 10

total_batches = len(train_loader)

for epoch in tqdm(range(num_epochs)):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()
    
    # tqdm 제거 후 단순히 에포크 시작을 출력합니다.
    print(f"Starting epoch {epoch + 1}/{num_epochs}")

    for step, (inputs, labels) in tqdm(enumerate(train_loader)):
        inputs, labels = inputs.to(device), labels.to(device)
        
        try:
            with autocast():  # 수정된 부분: 인자 없이 호출
                outputs = model(inputs).logits
                loss = criterion(outputs, labels)
                loss /= accumulation_steps

            scaler.scale(loss).backward()

            if (step + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            train_loss += loss.item()

            del inputs, labels  # outputs는 필요 없으므로 삭제하지 않음.
            torch.cuda.empty_cache()  # GPU 메모리 정리
            
            # 각 배치의 손실 값을 출력합니다.
            # print(f"Batch {step + 1}/{total_batches}, Loss: {loss.item():.4f}")
        
        except RuntimeError as e:
            if "out of memory" in str(e):
                print("CUDA out of memory. Skipping this batch.")
                torch.cuda.empty_cache()  # 메모리 정리 후 계속 진행.
                continue
            else:
                raise e

    avg_train_loss = train_loss / total_batches 
    print(f"Epoch {epoch + 1}/{num_epochs}, Average Loss: {avg_train_loss:.4f}")

# 학습된 가중치 저장
model.save_pretrained("/home/wangan/wav2vec_finetune/parameter/pretrained_wav2vec2_model")
processor.save_pretrained("/home/wangan/wav2vec_finetune/parameter/pretrained_wav2vec2_model_processor")
print("모델과 프로세서가 저장되었습니다.")


Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at kresnik/wav2vec2-large-xlsr-korean and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_51504/3102193497.py:105: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
  0%|          | 0/10 [00:00<?, ?it/s]

Starting epoch 1/10


/tmp/ipykernel_51504/3102193497.py:125: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # 수정된 부분: 인자 없이 호출


CUDA out of memory. Skipping this batch.


CUDA out of memory. Skipping this batch.


...

CUDA out of memory. Skipping this batch.


CUDA out of memory. Skipping this batch.


CUDA out of memory. Skipping this batch.


20132it [4:02:40,  1.38it/s]
100%|██████████| 10/10 [40:36:09<00:00, 14616.97s/it] 


Epoch 10/10, Average Loss: 0.0091
모델과 프로세서가 저장되었습니다.
